# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

In [1]:
import requests

# start a spark session and create a spark context
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .getOrCreate()

sc = spark.sparkContext

In [2]:
# set dl url for January 2019 trip data
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2019_trip_data, "wb") as f:
        f.write(response.content)


In [3]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)

# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [4]:
# Show the dataframe
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

In [5]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Inspect the dataset
df_trips.printSchema()

print("Number of trips:", df_trips.count())

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: integer (nullable = true)

Number of trips: 7696617


## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

In [6]:
#1 uniq key

df_trips = df_trips.withColumn(
    "trip_id",
    F.monotonically_increasing_id()
)

df_trips.select("trip_id", "tpep_pickup_datetime", "tpep_dropoff_datetime").show(10)

+-----------+--------------------+---------------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|
+-----------+--------------------+---------------------+
|60129542144| 2019-01-01 00:46:40|  2019-01-01 00:53:20|
|60129542145| 2019-01-01 00:59:47|  2019-01-01 01:18:59|
|60129542146| 2018-12-21 13:48:30|  2018-12-21 13:52:40|
|60129542147| 2018-11-28 15:52:25|  2018-11-28 15:55:45|
|60129542148| 2018-11-28 15:56:57|  2018-11-28 15:58:33|
|60129542149| 2018-11-28 16:25:49|  2018-11-28 16:28:26|
|60129542150| 2018-11-28 16:29:37|  2018-11-28 16:33:43|
|60129542151| 2019-01-01 00:21:28|  2019-01-01 00:28:37|
|60129542152| 2019-01-01 00:32:01|  2019-01-01 00:45:39|
|60129542153| 2019-01-01 00:57:32|  2019-01-01 01:09:32|
+-----------+--------------------+---------------------+
only showing top 10 rows


In [7]:
# 2 trip with highest passanger

df_trips.select(
    "trip_id",
    "passenger_count",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance"
).orderBy(
    F.desc("passenger_count")
).show(10, truncate=False)

+-----------+---------------+--------------------+---------------------+-------------+
|trip_id    |passenger_count|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|
+-----------+---------------+--------------------+---------------------+-------------+
|60130492100|9.0            |2019-01-05 13:12:29 |2019-01-05 13:12:32  |0.0          |
|60132426139|9.0            |2019-01-13 04:13:24 |2019-01-13 04:14:34  |0.0          |
|60130838431|9.0            |2019-01-07 03:19:36 |2019-01-07 03:20:01  |0.0          |
|60131554242|9.0            |2019-01-10 00:43:10 |2019-01-10 00:43:14  |0.0          |
|60134076851|9.0            |2019-01-19 16:45:25 |2019-01-19 16:45:27  |0.0          |
|60134394369|9.0            |2019-01-21 03:46:51 |2019-01-21 03:46:56  |0.0          |
|60134539934|9.0            |2019-01-21 19:20:28 |2019-01-21 19:20:31  |0.0          |
|60136828827|9.0            |2019-01-30 18:34:12 |2019-01-30 18:34:16  |0.0          |
|60136916020|9.0            |2019-01-30 22:

In [8]:
#3 average passenger

df_trips.select(
    F.avg("passenger_count").alias("average_passenger_count")
).show()

+-----------------------+
|average_passenger_count|
+-----------------------+
|     1.5670317144945614|
+-----------------------+



In [11]:
#4 min

df_trips.filter(
    F.col("trip_distance") > 0
).select(
    "trip_id",
    "trip_distance",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime"
).orderBy(
    F.asc("trip_distance")
).show(10, truncate=False)

+-----------+-------------+--------------------+---------------------+
|trip_id    |trip_distance|tpep_pickup_datetime|tpep_dropoff_datetime|
+-----------+-------------+--------------------+---------------------+
|60129562362|0.01         |2019-01-01 01:23:00 |2019-01-01 01:23:41  |
|60129577292|0.01         |2019-01-01 02:14:56 |2019-01-01 02:15:13  |
|60129563744|0.01         |2019-01-01 01:09:41 |2019-01-01 01:09:46  |
|60129562361|0.01         |2019-01-01 01:23:00 |2019-01-01 01:23:41  |
|60129564611|0.01         |2019-01-01 01:47:37 |2019-01-01 01:48:33  |
|60129556463|0.01         |2019-01-01 01:34:48 |2019-01-01 01:35:16  |
|60129565352|0.01         |2019-01-01 01:42:28 |2019-01-01 01:48:14  |
|60129560972|0.01         |2019-01-01 01:03:20 |2019-01-01 01:04:01  |
|60129565633|0.01         |2019-01-01 01:22:18 |2019-01-01 01:22:20  |
|60129567407|0.01         |2019-01-01 01:32:45 |2019-01-01 01:32:53  |
+-----------+-------------+--------------------+---------------------+
only s

In [10]:
#4 max

df_trips.select(
    "trip_id",
    "trip_distance",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime"
).orderBy(
    F.desc("trip_distance")
).show(10, truncate=False)

+-----------+-------------+--------------------+---------------------+
|trip_id    |trip_distance|tpep_pickup_datetime|tpep_dropoff_datetime|
+-----------+-------------+--------------------+---------------------+
|60135616235|831.8        |2019-01-25 21:56:39 |2019-01-25 22:06:08  |
|60133828777|700.7        |2019-01-18 16:32:24 |2019-01-18 16:39:20  |
|60136313129|214.01       |2019-01-28 17:24:11 |2019-01-29 04:37:16  |
|60134249678|211.36       |2019-01-20 12:22:24 |2019-01-20 17:05:36  |
|60134423929|201.27       |2019-01-21 10:13:05 |2019-01-21 13:28:01  |
|60134355479|160.52       |2019-01-20 20:45:50 |2019-01-20 23:26:38  |
|60132109587|144.2        |2019-01-11 23:04:43 |2019-01-11 23:28:43  |
|60134418563|143.63       |2019-01-21 09:50:57 |2019-01-21 13:05:04  |
|60130687083|142.88       |2019-01-06 09:58:04 |2019-01-06 12:35:10  |
|60134453458|132.8        |2019-01-21 12:13:22 |2019-01-21 14:41:29  |
+-----------+-------------+--------------------+---------------------+
only s

### Trip Distance Analysis

The dataset contains trips with a distance of 0 miles, which may correspond to cancelled trips, data recording issues, or trips where the taxi did not actually travel. For this reason, zero-distance trips are excluded when identifying the shortest meaningful trip.

The longest trips should also be interpreted carefully, as extremely large distances may represent outliers or data quality issues rather than typical taxi journeys.

In [16]:
#5 max time

df_trips = df_trips.withColumn(
    "trip_duration_minutes",
    F.timestamp_diff(
        "SECOND",
        F.col("tpep_pickup_datetime"),
        F.col("tpep_dropoff_datetime")
    ) / 60.0
)

df_trips.select(
    "trip_id",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_duration_minutes"
).show(5, truncate=False)

+-----------+--------------------+---------------------+---------------------+
|trip_id    |tpep_pickup_datetime|tpep_dropoff_datetime|trip_duration_minutes|
+-----------+--------------------+---------------------+---------------------+
|60129542144|2019-01-01 00:46:40 |2019-01-01 00:53:20  |6.666666666666667    |
|60129542145|2019-01-01 00:59:47 |2019-01-01 01:18:59  |19.2                 |
|60129542146|2018-12-21 13:48:30 |2018-12-21 13:52:40  |4.166666666666667    |
|60129542147|2018-11-28 15:52:25 |2018-11-28 15:55:45  |3.3333333333333335   |
|60129542148|2018-11-28 15:56:57 |2018-11-28 15:58:33  |1.6                  |
+-----------+--------------------+---------------------+---------------------+
only showing top 5 rows


In [17]:
#5 shorter without null or negative

df_trips.filter(
    F.col("trip_duration_minutes") > 0
).select(
    "trip_id",
    "trip_duration_minutes",
    "trip_distance",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime"
).orderBy(
    F.asc("trip_duration_minutes")
).show(10, truncate=False)

+-----------+---------------------+-------------+--------------------+---------------------+
|trip_id    |trip_duration_minutes|trip_distance|tpep_pickup_datetime|tpep_dropoff_datetime|
+-----------+---------------------+-------------+--------------------+---------------------+
|60129575663|0.016666666666666666 |0.0          |2019-01-01 02:18:39 |2019-01-01 02:18:40  |
|60129701309|0.016666666666666666 |0.0          |2019-01-01 19:39:51 |2019-01-01 19:39:52  |
|60129585711|0.016666666666666666 |0.0          |2019-01-01 03:25:47 |2019-01-01 03:25:48  |
|60129573664|0.016666666666666666 |0.0          |2019-01-01 02:15:41 |2019-01-01 02:15:42  |
|60129593682|0.016666666666666666 |0.0          |2019-01-01 03:40:05 |2019-01-01 03:40:06  |
|60129565495|0.016666666666666666 |0.0          |2019-01-01 01:09:17 |2019-01-01 01:09:18  |
|60129594319|0.016666666666666666 |1.65         |2019-01-01 03:31:50 |2019-01-01 03:31:51  |
|60129622151|0.016666666666666666 |0.09         |2019-01-01 10:34:43 |

In [18]:
#5 longer

df_trips.filter(
    F.col("trip_duration_minutes") > 0
).select(
    "trip_id",
    "trip_duration_minutes",
    "trip_distance",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime"
).orderBy(
    F.desc("trip_duration_minutes")
).show(10, truncate=False)

+-----------+---------------------+-------------+--------------------+---------------------+
|trip_id    |trip_duration_minutes|trip_distance|tpep_pickup_datetime|tpep_dropoff_datetime|
+-----------+---------------------+-------------+--------------------+---------------------+
|60129610411|43648.01666666667    |1.2          |2019-01-01 07:01:20 |2019-01-31 14:29:21  |
|60130134406|33856.683333333334   |1.1          |2019-01-03 22:24:36 |2019-01-27 10:41:17  |
|60130418000|31532.1              |3.9          |2019-01-05 04:21:40 |2019-01-27 01:53:46  |
|60133257869|7190.9               |0.0          |2019-01-16 15:31:14 |2019-01-21 15:22:08  |
|60131257409|1687.0333333333333   |0.78         |2019-01-08 20:18:52 |2019-01-10 00:25:54  |
|60132853285|1439.9833333333333   |18.61        |2019-01-14 23:24:05 |2019-01-15 23:24:04  |
|60137076918|1439.9666666666667   |7.39         |2019-01-31 14:20:52 |2019-02-01 14:20:50  |
|60132678464|1439.95              |0.0          |2019-01-14 10:01:52 |

### Trip Duration Analysis

Trips with a duration equal to or below 0 minutes are considered abnormal and are excluded from the shortest-trip analysis. They may result from data quality issues or incorrect timestamps.

Extremely long trip durations should also be treated as potential outliers, especially when they are inconsistent with the corresponding trip distance.

In [20]:
#6 busiest vs slowest day

df_trips = df_trips.withColumn(
    "pickup_date",
    F.to_date("tpep_pickup_datetime")
)

daily_trips = df_trips.groupBy(
    "pickup_date"
).count().orderBy(
    F.desc("count")
)

daily_trips.show(31)

+-----------+------+
|pickup_date| count|
+-----------+------+
| 2019-01-25|292499|
| 2019-01-11|291714|
| 2019-01-31|284625|
| 2019-01-17|284580|
| 2019-01-24|281959|
| 2019-01-10|281863|
| 2019-01-30|276774|
| 2019-01-16|272699|
| 2019-01-26|271993|
| 2019-01-15|267370|
| 2019-01-18|266848|
| 2019-01-12|265115|
| 2019-01-23|261151|
| 2019-01-29|259786|
| 2019-01-09|255868|
| 2019-01-22|255178|
| 2019-01-14|245082|
| 2019-01-28|241040|
| 2019-01-08|237310|
| 2019-01-05|236506|
| 2019-01-19|236365|
| 2019-01-04|236089|
| 2019-01-07|228816|
| 2019-01-13|227502|
| 2019-01-03|223965|
| 2019-01-27|220451|
| 2019-01-06|208823|
| 2019-01-20|203114|
| 2019-01-02|198737|
| 2019-01-21|192826|
| 2019-01-01|189432|
+-----------+------+
only showing top 31 rows


In [21]:
#6

january_2019 = df_trips.filter(
    (F.year("tpep_pickup_datetime") == 2019) &
    (F.month("tpep_pickup_datetime") == 1)
)

january_2019.groupBy(
    F.to_date("tpep_pickup_datetime").alias("date")
).count().orderBy(
    F.desc("count")
).show(31)

+----------+------+
|      date| count|
+----------+------+
|2019-01-25|292499|
|2019-01-11|291714|
|2019-01-31|284625|
|2019-01-17|284580|
|2019-01-24|281959|
|2019-01-10|281863|
|2019-01-30|276774|
|2019-01-16|272699|
|2019-01-26|271993|
|2019-01-15|267370|
|2019-01-18|266848|
|2019-01-12|265115|
|2019-01-23|261151|
|2019-01-29|259786|
|2019-01-09|255868|
|2019-01-22|255178|
|2019-01-14|245082|
|2019-01-28|241040|
|2019-01-08|237310|
|2019-01-05|236506|
|2019-01-19|236365|
|2019-01-04|236089|
|2019-01-07|228816|
|2019-01-13|227502|
|2019-01-03|223965|
|2019-01-27|220451|
|2019-01-06|208823|
|2019-01-20|203114|
|2019-01-02|198737|
|2019-01-21|192826|
|2019-01-01|189432|
+----------+------+



In [22]:
#7

df_trips = df_trips.withColumn(
    "pickup_hour",
    F.hour("tpep_pickup_datetime")
)

hourly_trips = df_trips.groupBy(
    "pickup_hour"
).count().orderBy(
    F.desc("count")
)

hourly_trips.show(24)

+-----------+------+
|pickup_hour| count|
+-----------+------+
|         18|515390|
|         19|475186|
|         17|468479|
|         15|452691|
|         14|433139|
|         20|423156|
|         16|420843|
|         21|409901|
|         13|404153|
|         12|401173|
|         11|375441|
|          8|373742|
|         22|369041|
|          9|365935|
|         10|361390|
|          7|304858|
|         23|281941|
|          0|207842|
|          6|178598|
|          1|149254|
|          2|109421|
|          3| 78086|
|          5| 75533|
|          4| 61424|
+-----------+------+



In [23]:
#8

df_trips = df_trips.withColumn(
    "time_of_day",
    F.when(
        (F.col("pickup_hour") >= 6) & (F.col("pickup_hour") < 12),
        "Morning"
    ).when(
        (F.col("pickup_hour") >= 12) & (F.col("pickup_hour") < 17),
        "Afternoon"
    ).when(
        (F.col("pickup_hour") >= 17) & (F.col("pickup_hour") < 22),
        "Evening"
    ).otherwise("Late Night")
)

df_trips.groupBy(
    "time_of_day"
).count().orderBy(
    F.desc("count")
).show()

+-----------+-------+
|time_of_day|  count|
+-----------+-------+
|    Evening|2292112|
|  Afternoon|2111999|
|    Morning|1959964|
| Late Night|1332542|
+-----------+-------+



In [25]:
#9

df_trips = df_trips.withColumn(
    "day_of_week",
    F.date_format("tpep_pickup_datetime", "EEEE")
)

In [26]:
#9

daily_counts = df_trips.groupBy(
    F.to_date("tpep_pickup_datetime").alias("date"),
    F.date_format("tpep_pickup_datetime", "EEEE").alias("day_of_week")
).count()

daily_counts.groupBy(
    "day_of_week"
).agg(
    F.avg("count").alias("average_trips")
).orderBy(
    F.desc("average_trips")
).show()

+-----------+------------------+
|day_of_week|     average_trips|
+-----------+------------------+
|   Thursday| 193863.2857142857|
|     Friday|155316.42857142858|
|   Saturday|144283.57142857142|
|  Wednesday|140584.88888888888|
|    Tuesday|          120908.4|
|     Sunday|        107488.125|
|     Monday|           90812.1|
+-----------+------------------+



In [27]:
#10

df_trips.select(
    F.corr("trip_distance", "tip_amount").alias("distance_tip_correlation")
).show()

+------------------------+
|distance_tip_correlation|
+------------------------+
|      0.5269200663652668|
+------------------------+



In [28]:
#10

df_distance_tip = df_trips.withColumn(
    "distance_category",
    F.when(F.col("trip_distance") < 2, "< 2 miles")
     .when(F.col("trip_distance") < 5, "2-5 miles")
     .when(F.col("trip_distance") < 10, "5-10 miles")
     .otherwise("10+ miles")
)

df_distance_tip.groupBy(
    "distance_category"
).agg(
    F.avg("tip_amount").alias("average_tip"),
    F.count("*").alias("number_of_trips")
).orderBy(
    F.asc("average_tip")
).show()

+-----------------+------------------+---------------+
|distance_category|       average_tip|number_of_trips|
+-----------------+------------------+---------------+
|        < 2 miles|1.1435660673705301|        4750401|
|        2-5 miles|1.9824044025647547|        1919063|
|       5-10 miles|3.4699004511456972|         585843|
|        10+ miles| 6.219343545352378|         441310|
+-----------------+------------------+---------------+



In [29]:
#11

df_trips.groupBy(
    "passenger_count"
).agg(
    F.avg("tip_amount").alias("average_tip"),
    F.count("*").alias("number_of_trips")
).orderBy(
    "passenger_count"
).show()

+---------------+--------------------+---------------+
|passenger_count|         average_tip|number_of_trips|
+---------------+--------------------+---------------+
|           NULL|0.061789899553571406|          28672|
|            0.0|  1.7869007761051638|         117381|
|            1.0|  1.8283524429075058|        5456515|
|            2.0|  1.8339324029045228|        1113894|
|            3.0|  1.7955889568213272|         314692|
|            4.0|  1.7027097823846875|         140753|
|            5.0|  1.8698681146978595|         323842|
|            6.0|  1.8568302035247934|         200811|
|            7.0|   6.542631578947368|             19|
|            8.0|   6.480689655172414|             29|
|            9.0|  3.1166666666666667|              9|
+---------------+--------------------+---------------+



In [30]:
#11

df_trips.select(
    F.corr("passenger_count", "tip_amount").alias("passenger_tip_correlation")
).show()

+-------------------------+
|passenger_tip_correlation|
+-------------------------+
|     0.001084223312167...|
+-------------------------+



In [31]:
#12

df_trips.select(
    "trip_id",
    "extra",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "fare_amount"
).orderBy(
    F.desc("extra")
).show(10, truncate=False)

+-----------+------+--------------------+---------------------+-------------+-----------+
|trip_id    |extra |tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|fare_amount|
+-----------+------+--------------------+---------------------+-------------+-----------+
|60134865627|535.38|2019-01-23 08:58:09 |2019-01-23 08:58:09  |0.0          |355676.98  |
|60136995374|23.04 |2019-01-31 10:06:09 |2019-01-31 10:06:09  |0.0          |4.5        |
|60130085347|18.5  |2019-01-03 18:32:36 |2019-01-03 19:45:29  |16.6         |61.0       |
|60129853196|18.5  |2019-01-02 16:33:28 |2019-01-02 17:17:08  |17.23        |52.0       |
|60130090452|18.5  |2019-01-03 18:19:33 |2019-01-03 18:55:54  |16.74        |47.5       |
|60131997230|18.5  |2019-01-11 16:08:48 |2019-01-11 16:58:11  |15.78        |49.0       |
|60133559279|18.5  |2019-01-17 16:24:12 |2019-01-17 17:29:50  |34.39        |91.5       |
|60129676693|18.5  |2019-01-01 16:09:32 |2019-01-01 16:47:57  |13.4         |39.5       |
|601328671

In [32]:
#13

df_trips.select(
    "passenger_count",
    "trip_distance",
    "trip_duration_minutes",
    "fare_amount",
    "tip_amount",
    "extra"
).summary().show()

+-------+------------------+------------------+---------------------+-----------------+------------------+------------------+
|summary|   passenger_count|     trip_distance|trip_duration_minutes|      fare_amount|        tip_amount|             extra|
+-------+------------------+------------------+---------------------+-----------------+------------------+------------------+
|  count|           7667945|           7696617|              7696617|          7696617|           7696617|           7696617|
|   mean|1.5670317144945614|2.8301461681153532|   16.551081570422276|12.52967677747685|1.8208300763883147|0.3374054146126797|
| stddev|1.2244198591042095| 3.774548394256295|    81.67539611217202|261.5897471783846|2.4994631914320986|0.5313564053935059|
|    min|               0.0|               0.0|             -84280.5|           -362.0|             -63.5|             -60.0|
|    25%|               1.0|               0.9|                  6.1|              6.0|               0.0|            

In [33]:
#13

df_trips.filter(
    (F.col("trip_distance") < 0) |
    (F.col("fare_amount") < 0) |
    (F.col("trip_duration_minutes") <= 0) |
    (F.col("passenger_count") > 6) |
    (F.col("trip_distance") > 100)
).select(
    "trip_id",
    "passenger_count",
    "trip_distance",
    "trip_duration_minutes",
    "fare_amount",
    "tip_amount"
).show(50, truncate=False)

+-----------+---------------+-------------+---------------------+-----------+----------+
|trip_id    |passenger_count|trip_distance|trip_duration_minutes|fare_amount|tip_amount|
+-----------+---------------+-------------+---------------------+-----------+----------+
|60129542172|3.0            |0.0          |0.0                  |6.5        |0.0       |
|60129542807|2.0            |0.1          |0.65                 |-2.5       |0.0       |
|60129542825|1.0            |0.0          |0.0                  |2.5        |0.0       |
|60129544546|1.0            |4.13         |26.616666666666667   |-19.0      |0.0       |
|60129544685|1.0            |1.35         |11.2                 |-8.5       |0.0       |
|60129544688|1.0            |0.0          |0.16666666666666666  |-2.5       |0.0       |
|60129544691|1.0            |0.16         |1.4                  |-3.0       |0.0       |
|60129545372|4.0            |0.42         |4.45                 |-4.5       |0.0       |
|60129545696|1.0     

### Outlier and Data Quality Analysis

Several potentially abnormal observations can be identified in the dataset:

- Some trips have a distance of 0 miles or an unusually large distance.
- Some trips have zero, negative, or extremely long durations.
- Negative fare amounts can be observed, which are not representative of a standard completed taxi trip.
- Some passenger counts may also be unusually high.

These observations should not automatically be considered errors. For example, negative fares may correspond to refunds or corrected transactions. However, they can significantly affect statistical analyses such as averages, minimum values, and correlations.

For a production data analysis or machine learning application, these records should therefore be investigated and potentially filtered according to clearly defined business rules.

### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough 
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

In [36]:
#14

zone_lookup_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

response = requests.get(zone_lookup_url)

zone_lookup_file = "taxi_zone_lookup.csv"

if response.status_code == 200:
    with open(zone_lookup_file, "wb") as f:
        f.write(response.content)

print("Download status:", response.status_code)

Download status: 200


In [37]:
#15

df_zones = spark.read.csv(
    zone_lookup_file,
    header=True,
    inferSchema=True
)

df_zones.show()
df_zones.printSchema()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [38]:
#16

pickup_zones = df_zones.select(
    F.col("LocationID").alias("PULocationID"),
    F.col("Borough").alias("pickup_borough"),
    F.col("Zone").alias("pickup_zone")
)

dropoff_zones = df_zones.select(
    F.col("LocationID").alias("DOLocationID"),
    F.col("Borough").alias("dropoff_borough"),
    F.col("Zone").alias("dropoff_zone")
)

df_enriched = df_trips \
    .join(pickup_zones, on="PULocationID", how="left") \
    .join(dropoff_zones, on="DOLocationID", how="left")

df_enriched.select(
    "trip_id",
    "pickup_borough",
    "pickup_zone",
    "dropoff_borough",
    "dropoff_zone"
).show(10)

+-----------+--------------+--------------------+---------------+--------------------+
|    trip_id|pickup_borough|         pickup_zone|dropoff_borough|        dropoff_zone|
+-----------+--------------+--------------------+---------------+--------------------+
|60129542144|     Manhattan|    Manhattan Valley|      Manhattan|Upper West Side S...|
|60129542145|     Manhattan|Upper West Side S...|      Manhattan|West Chelsea/Huds...|
|60129542146|     Manhattan|Upper East Side N...|      Manhattan|Upper East Side N...|
|60129542147|        Queens|Queensbridge/Rave...|         Queens|Queensbridge/Rave...|
|60129542148|        Queens|Queensbridge/Rave...|         Queens|Queensbridge/Rave...|
|60129542149|        Queens|Queensbridge/Rave...|         Queens|Queensbridge/Rave...|
|60129542150|        Queens|Queensbridge/Rave...|         Queens|Queensbridge/Rave...|
|60129542151|     Manhattan|       Midtown North|      Manhattan|Sutton Place/Turt...|
|60129542152|     Manhattan|Sutton Place/Tu

In [39]:
#17

df_enriched.groupBy(
    "pickup_borough"
).count().orderBy(
    F.desc("count")
).show()

+--------------+-------+
|pickup_borough|  count|
+--------------+-------+
|     Manhattan|6950965|
|        Queens| 471173|
|       Unknown| 159815|
|      Brooklyn|  91905|
|         Bronx|  18062|
|           N/A|   3890|
|           EWR|    446|
| Staten Island|    361|
+--------------+-------+



In [40]:
#18

df_enriched.groupBy(
    "dropoff_borough"
).count().orderBy(
    F.desc("count")
).show()

+---------------+-------+
|dropoff_borough|  count|
+---------------+-------+
|      Manhattan|6817355|
|         Queens| 340972|
|       Brooklyn| 301105|
|        Unknown| 149097|
|          Bronx|  58085|
|            N/A|  16904|
|            EWR|  10914|
|  Staten Island|   2185|
+---------------+-------+



In [41]:
#19

df_enriched.groupBy(
    "pickup_borough",
    "pickup_hour"
).count().orderBy(
    "pickup_borough",
    F.desc("count")
).show(200)

+--------------+-----------+------+
|pickup_borough|pickup_hour| count|
+--------------+-----------+------+
|         Bronx|          7|  1803|
|         Bronx|          8|  1445|
|         Bronx|          6|  1301|
|         Bronx|          9|  1158|
|         Bronx|         10|  1079|
|         Bronx|         14|  1014|
|         Bronx|         12|   956|
|         Bronx|         13|   918|
|         Bronx|         15|   897|
|         Bronx|         11|   885|
|         Bronx|         17|   812|
|         Bronx|         16|   756|
|         Bronx|         18|   741|
|         Bronx|          5|   736|
|         Bronx|         19|   519|
|         Bronx|         20|   440|
|         Bronx|         23|   410|
|         Bronx|         21|   408|
|         Bronx|          4|   403|
|         Bronx|         22|   353|
|         Bronx|          0|   324|
|         Bronx|          1|   254|
|         Bronx|          3|   225|
|         Bronx|          2|   225|
|      Brooklyn|          8|

In [42]:
#19

borough_hour_counts = df_enriched.groupBy(
    "pickup_borough",
    "pickup_hour"
).count()

window_busy = Window.partitionBy("pickup_borough").orderBy(F.desc("count"))
window_slow = Window.partitionBy("pickup_borough").orderBy(F.asc("count"))

busy_hours = borough_hour_counts.withColumn(
    "rank",
    F.row_number().over(window_busy)
).filter(
    F.col("rank") == 1
).select(
    "pickup_borough",
    F.col("pickup_hour").alias("busiest_hour"),
    F.col("count").alias("trips")
)

slow_hours = borough_hour_counts.withColumn(
    "rank",
    F.row_number().over(window_slow)
).filter(
    F.col("rank") == 1
).select(
    "pickup_borough",
    F.col("pickup_hour").alias("slowest_hour"),
    F.col("count").alias("trips")
)

busy_hours.show()
slow_hours.show()

+--------------+------------+------+
|pickup_borough|busiest_hour| trips|
+--------------+------------+------+
|         Bronx|           7|  1803|
|      Brooklyn|           8|  6935|
|           EWR|          15|    54|
|     Manhattan|          18|471539|
|           N/A|          19|   214|
|        Queens|          16| 29885|
| Staten Island|           8|    36|
|       Unknown|          18| 10751|
+--------------+------------+------+

+--------------+------------+-----+
|pickup_borough|slowest_hour|trips|
+--------------+------------+-----+
|         Bronx|           3|  225|
|      Brooklyn|           3| 1919|
|           EWR|          23|    1|
|     Manhattan|           4|53447|
|           N/A|           6|   88|
|        Queens|           3| 3085|
| Staten Island|           1|    3|
|       Unknown|           4| 1465|
+--------------+------------+-----+



In [43]:
#20

borough_day_counts = df_enriched.groupBy(
    "pickup_borough",
    "day_of_week"
).count()

window_day = Window.partitionBy(
    "pickup_borough"
).orderBy(
    F.desc("count")
)

borough_day_counts.withColumn(
    "rank",
    F.row_number().over(window_day)
).filter(
    F.col("rank") == 1
).select(
    "pickup_borough",
    F.col("day_of_week").alias("busiest_day"),
    F.col("count").alias("trips")
).show()

+--------------+-----------+-------+
|pickup_borough|busiest_day|  trips|
+--------------+-----------+-------+
|         Bronx|   Thursday|   3121|
|      Brooklyn|    Tuesday|  15779|
|           EWR|  Wednesday|     83|
|     Manhattan|   Thursday|1229554|
|           N/A|    Tuesday|    703|
|        Queens|   Thursday|  78972|
| Staten Island|     Friday|     64|
|       Unknown|   Thursday|  28929|
+--------------+-----------+-------+



In [44]:
#21

df_enriched.groupBy(
    "pickup_borough"
).agg(
    F.avg("trip_distance").alias("average_trip_distance")
).orderBy(
    F.desc("average_trip_distance")
).show()

+--------------+---------------------+
|pickup_borough|average_trip_distance|
+--------------+---------------------+
| Staten Island|   12.503601108033246|
|        Queens|   11.283218499361993|
|         Bronx|    7.233194552098303|
|      Brooklyn|    4.787677275447492|
|           N/A|    3.193850899742941|
|           EWR|    2.641098654708519|
|       Unknown|    2.415464130400774|
|     Manhattan|   2.2286693358402596|
+--------------+---------------------+



In [45]:
#22

df_enriched.groupBy(
    "pickup_borough"
).agg(
    F.avg("fare_amount").alias("average_fare")
).orderBy(
    F.desc("average_fare")
).show()

+--------------+------------------+
|pickup_borough|      average_fare|
+--------------+------------------+
|           EWR| 76.24024663677126|
|           N/A|  59.5731593830335|
| Staten Island|45.289861495844896|
|        Queens| 35.14462651722029|
|         Bronx| 26.26890543682963|
|      Brooklyn|18.649132800172286|
|       Unknown|14.944423051653523|
|     Manhattan|10.792468572351568|
+--------------+------------------+



In [47]:
#23

df_enriched.select(
    "trip_id",
    "fare_amount",
    "pickup_borough",
    "pickup_zone",
    "dropoff_borough",
    "dropoff_zone"
).orderBy(
    F.desc("fare_amount")
).show(1, truncate=False)

+-----------+-----------+--------------+---------------------+---------------+------------+
|trip_id    |fare_amount|pickup_borough|pickup_zone          |dropoff_borough|dropoff_zone|
+-----------+-----------+--------------+---------------------+---------------+------------+
|60132041799|623259.86  |Manhattan     |Upper East Side South|Manhattan      |Flatiron    |
+-----------+-----------+--------------+---------------------+---------------+------------+
only showing top 1 row


In [48]:
#23 min

df_enriched.select(
    "trip_id",
    "fare_amount",
    "pickup_borough",
    "pickup_zone",
    "dropoff_borough",
    "dropoff_zone"
).orderBy(
    F.asc("fare_amount")
).show(1, truncate=False)

+-----------+-----------+--------------+-----------+---------------+------------+
|trip_id    |fare_amount|pickup_borough|pickup_zone|dropoff_borough|dropoff_zone|
+-----------+-----------+--------------+-----------+---------------+------------+
|60134432793|-362.0     |Queens        |JFK Airport|Queens         |JFK Airport |
+-----------+-----------+--------------+-----------+---------------+------------+
only showing top 1 row


In [49]:
#24

download_url_2025 = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet"

response = requests.get(download_url_2025)

jan_2025_trip_data = "yellow_tripdata_2025-01.parquet"

if response.status_code == 200:
    with open(jan_2025_trip_data, "wb") as f:
        f.write(response.content)

print("Download status:", response.status_code)

Download status: 200


In [50]:
#24

df_trips_2025 = spark.read.parquet(jan_2025_trip_data)

df_trips_2025.show(5)
df_trips_2025.printSchema()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2025-01-01 00:18:38|  2025-01-01 00:26:59|              1|          1.6|         1|                 N|         229|    

In [51]:
#25

metrics_2019 = df_trips.agg(
    F.avg("passenger_count").alias("avg_passengers"),
    F.avg("trip_distance").alias("avg_distance"),
    F.avg("fare_amount").alias("avg_fare"),
    F.avg("tip_amount").alias("avg_tip")
).withColumn(
    "year",
    F.lit(2019)
)

metrics_2025 = df_trips_2025.agg(
    F.avg("passenger_count").alias("avg_passengers"),
    F.avg("trip_distance").alias("avg_distance"),
    F.avg("fare_amount").alias("avg_fare"),
    F.avg("tip_amount").alias("avg_tip")
).withColumn(
    "year",
    F.lit(2025)
)

comparison = metrics_2019.unionByName(metrics_2025)

comparison.select(
    "year",
    "avg_passengers",
    "avg_distance",
    "avg_fare",
    "avg_tip"
).show()

+----+------------------+------------------+-----------------+------------------+
|year|    avg_passengers|      avg_distance|         avg_fare|           avg_tip|
+----+------------------+------------------+-----------------+------------------+
|2019|1.5670317144945614|2.8301461681153532|12.52967677747685|1.8208300763883147|
|2025|1.2978589658806226| 5.855126178843539|17.08180276045484|2.9598127862758044|
+----+------------------+------------------+-----------------+------------------+



### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

In [54]:
#26 Dataframe as SQL table

df_trips.createOrReplaceTempView("trips")
df_zones.createOrReplaceTempView("zones")

In [55]:
#27 SQL - Average passenger count

spark.sql("""
SELECT
    AVG(passenger_count) AS average_passenger_count
FROM trips
""").show()

+-----------------------+
|average_passenger_count|
+-----------------------+
|     1.5670317144945614|
+-----------------------+



In [57]:
#28 SQL - Busiest hour

spark.sql("""
SELECT
    HOUR(tpep_pickup_datetime) AS pickup_hour,
    COUNT(*) AS number_of_trips
FROM trips
GROUP BY HOUR(tpep_pickup_datetime)
ORDER BY number_of_trips DESC
""").show()

+-----------+---------------+
|pickup_hour|number_of_trips|
+-----------+---------------+
|         18|         515390|
|         19|         475186|
|         17|         468479|
|         15|         452691|
|         14|         433139|
|         20|         423156|
|         16|         420843|
|         21|         409901|
|         13|         404153|
|         12|         401173|
|         11|         375441|
|          8|         373742|
|         22|         369041|
|          9|         365935|
|         10|         361390|
|          7|         304858|
|         23|         281941|
|          0|         207842|
|          6|         178598|
|          1|         149254|
+-----------+---------------+
only showing top 20 rows


In [58]:
#29 SQL - Pickups by borough with Join

spark.sql("""
SELECT
    z.Borough,
    COUNT(*) AS number_of_pickups
FROM trips t
JOIN zones z
    ON t.PULocationID = z.LocationID
GROUP BY z.Borough
ORDER BY number_of_pickups DESC
""").show()

+-------------+-----------------+
|      Borough|number_of_pickups|
+-------------+-----------------+
|    Manhattan|          6950965|
|       Queens|           471173|
|      Unknown|           159815|
|     Brooklyn|            91905|
|        Bronx|            18062|
|          N/A|             3890|
|          EWR|              446|
|Staten Island|              361|
+-------------+-----------------+



# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing